# Potter Airlines Dynamic Pricing and Booking System

This notebook builds the pricing logic, stores flight data in SQLite, performs vectorized pricing analysis, and demonstrates an interactive booking workflow.


## 1. Setup and Load Initial Data


### 1.1 Load CSV


In [ ]:
import sqlite3
import math
from numbers import Real
from contextlib import closing
import pandas as pd

flights = pd.read_csv("flights.csv")

def load_flights_db(db_path="potter_airlines.db"):
    """
    Connect to the SQLite database and return the full flights table as a DataFrame.
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query("SELECT * FROM flights", conn)
    return df


## 2. Pricing Helper Function


In [ ]:

def get_time_factor(days):
    if days < 0:
        raise ValueError('Day before dep cannot smaller than 0')
    if days <= 7:
        time_factor = 1.35
    elif days <= 21:
        time_factor = 1.1
    else:
        time_factor = 1
    return time_factor

print(get_time_factor(12))
try:
    get_time_factor(-1)
except ValueError:
    print("Invalid value: days must be 0 or greater.")
print(get_time_factor(0))



## 3. Flight Class


In [ ]:
def require_integer(value, name):
    """Accept whole numeric values without silently truncating fractions."""
    if (isinstance(value, bool) or not isinstance(value, Real)
            or not math.isfinite(value) or value != int(value)):
        raise ValueError(f"{name} must be a whole number.")
    return int(value)


class Flight:
    def __init__(self, flight_id, base_fare, seats_remaining, capacity, demand_factor, seasonal_factor, origin, destination):

        capacity = require_integer(capacity, "Capacity")
        seats_remaining = require_integer(seats_remaining, "Seats remaining")
        for name, value in [("Base fare", base_fare), ("Demand factor", demand_factor),
                            ("Seasonal factor", seasonal_factor)]:
            if not isinstance(value, Real) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and greater than 0.")

        # Check ranges before storing values.
        if capacity <= 0:
            raise ValueError('Capacity must be greater than 0')
        if not 0 <= seats_remaining <= capacity:
            raise ValueError('Seat remaining must within 0 and Capacity ')
        if base_fare <= 0:
            raise ValueError('Base fare must greater than 0')
        if demand_factor <= 0 or seasonal_factor <= 0:
            raise ValueError('Demand and seasonal factors must be greater than 0')

        self.flight_id = str(flight_id)
        self.base_fare = float(base_fare)
        self.seats_remaining = int(seats_remaining)
        self.capacity = int(capacity)
        self.demand_factor = float(demand_factor)
        self.seasonal_factor = float(seasonal_factor)
        self.origin = origin
        self.destination = destination

    def load_factor(self):
        return 1 - self.seats_remaining/self.capacity

    def update_seats(self, change):
        change = require_integer(change, "Seat change")
        new_value = self.seats_remaining + change

        if not 0 <= new_value <= self.capacity:
            raise ValueError("seat update is outside 0..capacity")
        else:
            self.seats_remaining = new_value

    def calculate_price(self, days):
        time_factor = get_time_factor(days)
        capacity_factor = 1 + 0.45 * self.load_factor()
        price = self.base_fare * time_factor * capacity_factor * self.demand_factor * self.seasonal_factor
        return round(min(1500,max(45,price)), 2)

    def to_dict(self):
        return vars(self).copy()


test_flight = Flight('PA1001', 300, 18, 180, 1.2, 1.0, 'YYZ', 'YVR')
print(test_flight.destination)

### 3.1 Flight Class Validation


In [ ]:
print(test_flight.seats_remaining)
print(test_flight.load_factor())
try:
    test_flight.update_seats(-20)
except ValueError:
    print("Invalid seat update blocked.")

In [ ]:
old_price = test_flight.calculate_price(12)
test_flight.update_seats(-5)
new_price = test_flight.calculate_price(12)
print(old_price)
print(new_price)

In [ ]:
try:
    bad_flight = Flight("PA_TEST", 300, 181, 180, 1.2, 1.0, 'YYZ', 'CHN')
except ValueError:
    print("Invalid flight update blocked.")

## 4. SQLite Persistence

Run these cells in order from this notebook folder. On first use, the setup cell creates the flights table from CSV. If the table already exists, its saved bookings are preserved. Use the manual reset in section 6.2 to restore the CSV data explicitly.


### 4.1 Create Table and Insert Flights


In [ ]:
def setup_flights_table(flights_df, db_path="potter_airlines.db"):
    """Create the flights table and load one row per validated Flight."""
    
    # Validate everything before DROP TABLE; a bad CSV must not erase bookings.
    if flights_df.empty or flights_df.isna().any().any():
        raise ValueError("Flight data must be non-empty and contain no missing values.")
    if not flights_df["flight_id"].astype(str).is_unique:
        raise ValueError("Flight IDs must be unique.")
    dates = pd.to_datetime(flights_df["departure_date"], errors="coerce")
    if dates.isna().any():
        raise ValueError("Every flight must have a valid departure date.")
    validated_records = []
    for record in flights_df.to_dict(orient="records"):
        flight = Flight(
            record["flight_id"], record["base_fare"], record["seats_remaining"],
            record["capacity"], record["demand_factor"], record["seasonal_factor"],
            record["origin"], record["destination"]
        )
        flight_dict = flight.to_dict()
        validated_records.append((flight_dict, record["departure_date"]))

    with closing(sqlite3.connect(db_path)) as conn, conn:
        cursor = conn.cursor()

        # Explicit BEGIN also puts DROP/CREATE inside the rollback boundary.
        cursor.execute("BEGIN")
        cursor.execute("DROP TABLE IF EXISTS flights")
        cursor.execute("""
            CREATE TABLE flights (
                flight_id TEXT PRIMARY KEY,
                origin TEXT NOT NULL,
                destination TEXT NOT NULL,
                departure_date TEXT NOT NULL,
                base_fare REAL NOT NULL,
                capacity INTEGER NOT NULL,
                seats_remaining INTEGER NOT NULL,
                demand_factor REAL NOT NULL,
                seasonal_factor REAL NOT NULL
            )
        """)

        for flight_dict, departure_date in validated_records:
            cursor.execute(
                """
                INSERT INTO flights (
                    flight_id, origin, destination, departure_date,
                    base_fare, capacity, seats_remaining, demand_factor, seasonal_factor
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    flight_dict["flight_id"], flight_dict["origin"],
                    flight_dict["destination"], departure_date,
                    flight_dict["base_fare"], flight_dict["capacity"],
                    flight_dict["seats_remaining"], flight_dict["demand_factor"],
                    flight_dict["seasonal_factor"]
                )
            )

        conn.commit()
    conn.close()

def initialize_flights_database(flights_df, db_path="potter_airlines.db"):
    """Load CSV on first use only; preserve existing bookings on Run All."""
    with closing(sqlite3.connect(db_path)) as conn:
        exists = conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = 'flights'"
        ).fetchone()
    if exists is None:
        setup_flights_table(flights_df, db_path)
        print("CSV flights validated and inserted into a new database.")
    else:
        print("Using existing flights and saved bookings.")


initialize_flights_database(flights)

### 4.2 Query Flights


In [ ]:
def get_flights_by_destination(destination, db_path="potter_airlines.db"):
    """
    Return every stored flight row to the given destination 
    using a parameterized SELECT query.
    """
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT * FROM flights WHERE destination = ?",
            (destination,)
        )
        rows = cursor.fetchall()
    conn.close()
    return rows

destination = "YVR"
rows = get_flights_by_destination(destination)
print(f"Flights to {destination}:", rows)

### 4.3 Update and Delete a Test Flight


In [ ]:
def update_db_seats(flight_id, new_seats, db_path="potter_airlines.db"):
    new_seats = require_integer(new_seats, "Seats remaining")
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT capacity FROM flights WHERE flight_id = ?",
            (flight_id,)
        )
        stored_flight = cursor.fetchone()

        if stored_flight is None:
            raise ValueError(f"Flight {flight_id} was not found.")

        capacity = stored_flight[0]
        if not 0 <= new_seats <= capacity:
            raise ValueError(f"Seats remaining must be between 0 and {capacity}.")

        cursor.execute(
            "UPDATE flights SET seats_remaining = ? WHERE flight_id = ?",
            (new_seats, flight_id)
        )
        conn.commit()


def run_crud_test(db_path="potter_airlines.db"):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        cursor.execute("DELETE FROM flights WHERE flight_id = ?", ("SQL_DEMO",))
        cursor.execute(
            "INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            ("SQL_DEMO", "YYZ", "YVR", "2026-10-01", 300, 180, 18, 1.2, 1.0)
        )
        conn.commit()
    conn.close()
    
    update_db_seats("SQL_DEMO", 13, db_path)
    
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT flight_id, seats_remaining FROM flights WHERE flight_id = ?",
            ("SQL_DEMO",)
        )
        updated = cursor.fetchall()
        print("Updated test flight:", updated)
        assert updated == [("SQL_DEMO", 13)]
        
        cursor.execute("DELETE FROM flights WHERE flight_id = ?", ("SQL_DEMO",))
        cursor.execute("SELECT * FROM flights WHERE flight_id = ?", ("SQL_DEMO",))
        deleted = cursor.fetchall()
        assert deleted == []
        print("Test flight deleted:", deleted)
        conn.commit()
    conn.close()


run_crud_test()


### 4.4 Verify Stored Data


In [ ]:
# Reuse the globally defined load function!
stored_flights = load_flights_db("potter_airlines.db")

# Verify the data was loaded correctly
assert len(stored_flights) == len(flights)
print(f"Stored flight count: {len(stored_flights)}")
print(stored_flights.head())

## 5. Pricing All Flights (Vectorized)

The SQLite section above loads the flights but does not price them yet. This section:

1. Prepares the departure dates and checks the whole table before calculating anything.
2. Prices every flight at once using Pandas/NumPy (no `for` loop), using the same formula as `Flight.calculate_price`.
3. Filters out sold-out flights and ranks the rest.
4. Updates one flight's seats and shows the price respond, persisting the change to SQLite.
5. Demonstrates one edge case on real data (selling a seat on an already sold-out flight).

`quote_date` is fixed so the notebook gives the same numbers every time it is run.

### 5.1 Prepare dates and check the table
`departure_date` is stored as text, so it is converted to an actual date first. `days_until_departure` is how many days from `quote_date` until that flight leaves.

The `assert` lines repeat, on the whole table at once, the same assumptions the `Flight` class already checks one flight at a time.

In [ ]:
def prepare_and_validate_flights(df, quote_date_str="2026-09-19", verbose=True):
    """
    Process departure dates, calculate days until departure, validate inputs, 
    and remove already departed flights.
    """
    priced = df.copy()
    quote_date = pd.Timestamp(quote_date_str)
    
    priced["departure_date"] = pd.to_datetime(priced["departure_date"])
    priced["days_until_departure"] = (priced["departure_date"] - quote_date).dt.days

    assert priced.notna().all().all()
    assert priced["flight_id"].is_unique
    for column in ["capacity", "seats_remaining"]:
        for value in priced[column]:
            require_integer(value, column)
    assert (priced["capacity"] > 0).all()
    assert priced["seats_remaining"].between(0, priced["capacity"]).all()
    assert (priced["base_fare"] > 0).all()
    assert (priced["demand_factor"] > 0).all()
    assert (priced["seasonal_factor"] > 0).all()
    
    departed = priced.loc[priced["days_until_departure"] < 0, "flight_id"].tolist()
    valid_flights = priced[priced["days_until_departure"] >= 0].copy()
    
    # Only print when verbose is True for testing
    if verbose:
        print("Input checks passed.")
        print("Already departed, excluded from pricing:", departed)
    
    return valid_flights

prepared_flights = prepare_and_validate_flights(stored_flights, quote_date_str="2026-09-19")

### 5.2 Price every flight at once
This is the same formula as `Flight.calculate_price`, but applied to the whole column instead of one flight:

```text
price = base_fare x time_factor x capacity_factor x demand_factor x seasonal_factor
```

`np.select` picks the first true condition for `time_factor`, same three tiers as `get_time_factor`.

In [ ]:
import numpy as np

def calculate_vectorized_prices(df):
    """
    Calculate time, load, and capacity factors to compute the final price 
    using vectorized operations.
    """
    # Create a copy to prevent modifying the dataframe from the previous step
    priced = df.copy()
    
    days = priced["days_until_departure"]
    priced["time_factor"] = np.select(
        [days <= 7, days <= 21],
        [1.35, 1.10],
        default=1.0,
    )

    priced["load_factor"] = 1 - priced["seats_remaining"] / priced["capacity"]
    priced["capacity_factor"] = 1 + 0.45 * priced["load_factor"]

    priced["raw_price"] = (
        priced["base_fare"]
        * priced["time_factor"]
        * priced["capacity_factor"]
        * priced["demand_factor"]
        * priced["seasonal_factor"]
    )
    
    # Clip prices to be strictly between 45 and 1500, and round to 2 decimals
    priced["final_price"] = np.clip(priced["raw_price"], 45, 1500).round(2)

    # Validate that bounds are respected
    assert priced["final_price"].between(45, 1500).all()

    return priced

priced_flights = calculate_vectorized_prices(prepared_flights)
priced_flights[["flight_id", "origin", "destination", "days_until_departure",
                "seats_remaining", "final_price"]]

In [ ]:
def verify_pricing_consistency(df, target_flight_id="PA1002"):
    """
    Verify that the vectorized price perfectly matches 
    the object-oriented (Flight class) calculation.
    """
    # Extract the target flight from the dataframe
    check_row = df.loc[df["flight_id"] == target_flight_id].iloc[0]
    
    # Instantiate a single Flight object
    check_flight = Flight(
        check_row["flight_id"], check_row["base_fare"], check_row["seats_remaining"],
        check_row["capacity"], check_row["demand_factor"], check_row["seasonal_factor"],
        check_row["origin"], check_row["destination"],
    )
    
    # Calculate using the class method
    scalar_price = check_flight.calculate_price(int(check_row["days_until_departure"]))
    
    # Ensure the difference is less than a cent
    assert abs(scalar_price - check_row["final_price"]) < 0.01
    
    print(f"Vectorized and single-flight prices agree for {target_flight_id}: {scalar_price}")

verify_pricing_consistency(priced_flights, target_flight_id="PA1002")

### 5.3 Filter and rank flights
A sold-out flight (0 seats remaining) cannot be booked, so it is excluded before ranking. The remaining flights are ranked by `final_price`, cheapest first.

In [ ]:
def filter_and_rank_available_flights(df, verbose=True):
    """
    Filter out sold-out flights and rank the remaining flights by price.
    """
    sold_out_mask = df["seats_remaining"] == 0
    sold_out_ids = df.loc[sold_out_mask, "flight_id"].tolist()
    available = df[~sold_out_mask].copy()
    ranked_available = available.sort_values("final_price")
    
    # Only print when verbose is True for testing
    if verbose:
        print(f"Sold-out flights excluded: {len(sold_out_ids)}")
        if sold_out_ids:
            print(f"Sold-out flight IDs: {sold_out_ids}")
            
    return ranked_available

available_flights = filter_and_rank_available_flights(priced_flights)

# Extract the top 5 for display
cheapest_available = available_flights.head(5)
cheapest_available[["flight_id", "origin", "destination", 
                    "seats_remaining", "final_price"]]

### 5.4 Summarize available flights by route

Group available flights by origin and destination to compare flight counts, remaining seats, and ticket prices.

In [ ]:
flight_summary = available_flights.groupby(["origin", "destination"], as_index=False).agg(
    flight_count=("flight_id", "count"),
    total_seats_remaining=("seats_remaining", "sum"),
    average_price=("final_price", "mean"),
    lowest_price=("final_price", "min"),
).round(2)

flight_summary

### 5.5 Update an operational value and re-price (manual)
Uncomment the call below to sell 10 seats on PA1002. Each call reads the current inventory from SQLite and deducts seats in one transaction. The before/after fares use that current inventory. The call is commented out so Run All preserves saved bookings.


In [ ]:
def simulate_seat_sale(raw_df, priced_df, target_id, seats_to_sell, db_path="potter_airlines.db"):
    """Sell from current database inventory; raw_df is retained for call compatibility."""
    seats_to_sell = require_integer(seats_to_sell, "Tickets to sell")
    if seats_to_sell <= 0:
        raise ValueError("Tickets to sell must be greater than 0.")
    days = priced_df.loc[priced_df["flight_id"] == target_id, "days_until_departure"]
    if days.empty:
        raise ValueError("Flight is not available for pricing on this quote date.")
    days_to_dep = int(days.iloc[0])

    with closing(sqlite3.connect(db_path)) as conn, conn:
        # Keep the read, deduction, and read-back in one transaction.
        conn.execute("BEGIN IMMEDIATE")
        conn.row_factory = sqlite3.Row
        row = conn.execute("SELECT * FROM flights WHERE flight_id = ?", (target_id,)).fetchone()
        if row is None:
            raise ValueError(f"Flight {target_id} was not found.")
        flight = Flight(
            row["flight_id"], row["base_fare"], row["seats_remaining"], row["capacity"],
            row["demand_factor"], row["seasonal_factor"], row["origin"], row["destination"]
        )
        seats_before = flight.seats_remaining
        price_before = flight.calculate_price(days_to_dep)
        cursor = conn.execute(
            """UPDATE flights SET seats_remaining = seats_remaining - ?
               WHERE flight_id = ? AND seats_remaining >= ?""",
            (seats_to_sell, target_id, seats_to_sell)
        )
        if cursor.rowcount != 1:
            raise ValueError("Insufficient seats for this booking.")
        seats_after = conn.execute(
            "SELECT seats_remaining FROM flights WHERE flight_id = ?", (target_id,)
        ).fetchone()[0]
        flight.update_seats(-seats_to_sell)
        assert seats_after == flight.seats_remaining
        price_after = flight.calculate_price(days_to_dep)
        conn.commit()

    print(f"{target_id}: sold {seats_to_sell} seats, {seats_before} -> "
          f"{seats_after} seats remaining (confirmed in database).")
    print(f"Price moved from {price_before} to {price_after} as the load factor increased.")


# Run manually to sell 10 seats. Run All must not change saved bookings.
# simulate_seat_sale(stored_flights, priced_flights, target_id="PA1002", seats_to_sell=10)


### 5.6 Edge case: selling seats on a sold-out flight
`PA1007` has 0 seats remaining in `flights.csv`. `update_seats` should reject any attempt to sell a seat on it, the same guard already tested on `test_flight` earlier in this notebook, now shown on real data.

In [ ]:
def test_sold_out_edge_case(raw_df, target_id):
    """
    Verify that attempting to sell a seat on a sold-out flight 
    raises a ValueError and is properly blocked.
    """
    soldout_row = raw_df.loc[raw_df["flight_id"] == target_id].iloc[0]
    
    soldout_flight = Flight(
        soldout_row["flight_id"], soldout_row["base_fare"], soldout_row["seats_remaining"],
        soldout_row["capacity"], soldout_row["demand_factor"], soldout_row["seasonal_factor"],
        soldout_row["origin"], soldout_row["destination"],
    )
    
    print(f"{soldout_flight.flight_id} seats remaining: {soldout_flight.seats_remaining}")

    try:
        soldout_flight.update_seats(-1)
    except ValueError:
        print("Blocked: cannot sell a seat on a sold-out flight.")

# --- Execution for this cell ---
test_sold_out_edge_case(stored_flights, target_id="PA1007")

## 6. Interactive Booking System (Live)

This final section provides a command-line interface simulating a customer booking experience. It fetches the live state of the SQLite database, recalculates prices, and allows the user to purchase a ticket. Buying a ticket decreases `seats_remaining` in the database, so a later run uses the updated seat count and price.


In [ ]:
def get_ticket_count(max_available):
    """Ask for a valid number of tickets that does not exceed availability."""
    while True:
        ticket_input = input("How many tickets would you like to buy? ").strip()

        try:
            ticket_count = int(ticket_input)
        except ValueError:
            print("Please enter a positive whole number.")
            continue

        if ticket_count <= 0:
            print("The number of tickets must be greater than 0.")
            continue

        if ticket_count > max_available:
            print(f"Only {max_available} seats are available.")
            continue

        return ticket_count


def run_interactive_booking(db_path="potter_airlines.db", quote_date_str="2026-09-19"):

    """
    A live command-line interface to search for flights and book a seat.
    Fetches fresh data from SQLite, recalculates prices, and loops until 
    the user completes a booking or chooses to exit.
    """
    print("=== Potter Airlines Booking System (Live) ===")
    
    # 1. Fetch live data and run the processing pipeline
    raw_df = load_flights_db(db_path)
    prepared_df = prepare_and_validate_flights(raw_df, quote_date_str, verbose=False)
    priced_df = calculate_vectorized_prices(prepared_df)
    available_df = filter_and_rank_available_flights(priced_df, verbose=False)
    
    # Extract unique available destinations dynamically
    valid_destinations = available_df["destination"].unique().tolist()
    dest_prompt_str = ", ".join(valid_destinations)
    
    # 2. Interactive booking loop
    while True:
        # Prompt user with dynamically generated available airports
        user_dest = input(f"\nEnter destination ({dest_prompt_str}) or type 'EXIT','QUIT' to quit: ").strip().upper()
        
        # Exit condition 1: User types EXIT or QUIT
        if user_dest in ["EXIT", "QUIT"]:
            print("\nExiting booking system. Have a great day!")
            break
            
        options = available_df[available_df["destination"] == user_dest]
        
        # If user types an invalid destination, warn and restart loop
        if options.empty:
            print(f"Sorry! Invalid destination. Please choose from: {dest_prompt_str}")
            continue
            
        print(f"\nAvailable flights to {user_dest}:")
        print(options[["flight_id", "departure_date", "seats_remaining", "final_price"]].to_string(index=False))
        
        # Flight selection prompt
        choice = input("\nEnter flight ID to book (or press Enter to search again, 'EXIT' to quit): ").strip().upper()
        
        # Exit condition 2: User types EXIT here
        if choice in ["EXIT", "QUIT"]:
            print("\nExiting booking system. Have a great day!")
            break
            
        # If user just hits Enter, return to the start of the loop
        if not choice:
            print("\nReturning to search menu...")
            continue
            
        # Process booking if the flight ID is valid
        if choice in options["flight_id"].values:
            selected_flight = options.loc[options["flight_id"] == choice].iloc[0]
            ticket_count = get_ticket_count(int(selected_flight["seats_remaining"]))

            print("\nProcessing booking...")
            simulate_seat_sale(
                raw_df=raw_df,
                priced_df=priced_df,
                target_id=choice,
                seats_to_sell=ticket_count,
                db_path=db_path
            )
            print("\nYes! Booking confirmed! The database has been updated.")
            print("Run the booking demo again to see seats remaining decrease and the dynamic price increase!")
            break # Exit loop completely after a successful booking
        else:
            # Invalid flight ID, warn and restart loop
            print("\nSorry! Invalid flight ID. Returning to search menu...")
            continue


### 6.1 Run the Booking Demo Manually

Run the next cell when you want to use the interactive booking system. It is commented out so **Run All** can finish without waiting for keyboard input.


In [ ]:
# Remove the # below when you want to start the interactive booking demo.
# run_interactive_booking()


### 6.2 Optional Database Reset


In [ ]:
# --- Reset Database Button ---

def reset_database_to_initial_state(
    csv_path="flights.csv",
    db_path="potter_airlines.db"
):
    flights_df = pd.read_csv(csv_path)
    setup_flights_table(flights_df, db_path)
    print("Database reset to initial state from CSV. Ready for a new demo!")

# Run manually only when you want to discard booking changes.
# reset_database_to_initial_state()
